# Dataset and pipeline exploration

A walkthrough of each stage of the skin-lesion risk pipeline on a single image,
useful for sanity-checking a newly prepared dataset before training.

> **Not a medical device.** This is a student research prototype. Nothing here
> is a diagnosis.

Run `python data/make_demo_dataset.py` (or one of the `prepare_*.py` scripts)
first, so that `data/processed/unified.csv` exists.

In [ ]:
import sys
from pathlib import Path

# Make the repo importable regardless of where Jupyter was launched.
REPO = Path.cwd()
while not (REPO / "configs" / "default.yaml").exists() and REPO != REPO.parent:
    REPO = REPO.parent
sys.path.insert(0, str(REPO))

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.config import load_config

cfg = load_config()
print("config:", cfg.source)
print("risk classes:", cfg.get("labels.risk_classes"))

## 1. The unified dataset

Both source datasets are normalised into one CSV, so everything below works for
HAM10000, PAD-UFES-20, or both merged together.

In [ ]:
from src.dataset import class_distribution, load_unified_csv

df = load_unified_csv(cfg)
print(f"{len(df)} images from {sorted(df['source'].unique())}")
if "synthetic_demo" in set(df["source"]):
    print("\n*** SYNTHETIC DEMO DATA - anything measured below verifies the")
    print("*** code path, NOT clinical accuracy.")
df.head()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))

counts = class_distribution(df, cfg.get("labels.risk_classes"))
axes[0].bar(list(counts), list(counts.values()),
            color=["#2e7d32", "#ef6c00", "#c62828"])
axes[0].set_title("Risk class balance")
axes[0].set_ylabel("images")

df["site"].value_counts().plot.barh(ax=axes[1], color="#455a64")
axes[1].set_title("Anatomical site")

pd.to_numeric(df["age"], errors="coerce").dropna().plot.hist(
    bins=20, ax=axes[2], color="#455a64")
axes[2].set_title("Age distribution")
plt.tight_layout()

# Class imbalance is the thing to watch: HAM10000 is ~67% ordinary nevi, which
# is why training uses inverse-frequency class weights and why macro-F1 (not
# accuracy) is the model-selection metric.
imbalance = max(counts.values()) / max(min(counts.values()), 1)
print(f"imbalance ratio (largest / smallest class): {imbalance:.1f}x")

## 2. Splits are made over lesions, not images

HAM10000 contains several photos of the same physical lesion. Splitting
per-image would place near-duplicates in both train and test and inflate every
score.

In [ ]:
from src.dataset import group_aware_split

train_df, val_df, test_df = group_aware_split(df, cfg)
for name, part in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(f"{name:6s} {len(part):5d} images  "
          f"{len(part['lesion_id'].unique()):5d} lesions  "
          f"{class_distribution(part, cfg.get('labels.risk_classes'))}")

overlap = set(train_df["lesion_id"]) & set(test_df["lesion_id"])
print(f"\nlesions in BOTH train and test: {len(overlap)}   (must be 0)")

## 3. Image quality gate

This runs before anything else. A confident-looking number computed from a
blurred photo is the worst possible output for this kind of tool.

In [ ]:
from src.quality.image_quality import assess_image_quality
from src.utils import load_image_rgb

sample_path = load_config().resolve(df.iloc[0]["image_path"])
rgb = load_image_rgb(sample_path)

variants = {
    "original": rgb,
    "blurred": cv2.GaussianBlur(rgb, (31, 31), 0),
    "too dark": (rgb.astype(np.float32) * 0.08).astype(np.uint8),
    "over-exposed": np.clip(rgb.astype(np.float32) * 1.9 + 80, 0, 255).astype(np.uint8),
}

fig, axes = plt.subplots(1, len(variants), figsize=(16, 4.5))
for ax, (name, image) in zip(axes, variants.items()):
    report = assess_image_quality(image, cfg)
    ax.imshow(image)
    ax.axis("off")
    failed = ", ".join(c.name for c in report.failures) or "none"
    ax.set_title(f"{name}\npass={report.passed}\nfailed: {failed}", fontsize=9)
plt.tight_layout()

## 4. Segmentation

Classical and inspectable: flat-field the lesion-contrast channel, Otsu
threshold, pick the most lesion-like component, refine with GrabCut, fill
holes.

In [ ]:
from src.segmentation import segment_lesion

seg = segment_lesion(rgb)
print(f"found={seg.found}  area_fraction={seg.area_fraction:.3f}  bbox={seg.bbox}")

outline = rgb.copy()
contours, _ = cv2.findContours((seg.mask > 0).astype(np.uint8),
                               cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
cv2.drawContours(outline, contours, -1, (0, 255, 0), 2)

fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))
for ax, image, title in zip(axes, [rgb, seg.mask, outline],
                            ["photo", "mask", "outline"]):
    ax.imshow(image, cmap="gray" if image.ndim == 2 else None)
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()

## 5. ABCDE measurements

Computed from the mask **independently of the neural network**. That
independence is the point: if the model and these measurements disagree, the
report says so rather than inventing a justification for the model.

In [ ]:
from src.explain.abcde import compute_abcde

abcde = compute_abcde(rgb, cfg, mask=seg)
print(abcde.summary(), "\n")
for feature in abcde.features:
    if not feature.measurable:
        marker = "?"
    elif feature.flagged:
        marker = "!"
    else:
        marker = "-"
    score = f"{feature.score:.3f}" if feature.score is not None else "  n/a"
    print(f"[{marker}] {feature.letter}  {feature.title:12s} {score}")
    print(f"      {feature.sentence}\n")

In [ ]:
# How well the criteria separate the risk classes across the dataset.
rows = []
for record in df.sample(min(45, len(df)), random_state=0).to_dict("records"):
    image = load_image_rgb(load_config().resolve(record["image_path"]))
    report = compute_abcde(image, cfg)
    entry = {"risk": record["risk_label"], "flags": report.flag_count}
    for feature in report.features[:3]:      # A, B, C carry the signal
        entry[feature.letter] = feature.score
    rows.append(entry)

summary = pd.DataFrame(rows)
print(summary.groupby("risk")[["A", "B", "C", "flags"]].mean().round(3))

## 6. Risk prediction, Grad-CAM and the full report

Needs a trained checkpoint: run `python src/model/train.py` first. Without one,
`LesionAnalyzer(strict=False)` still runs, but its predictions are meaningless
and it says so.

In [ ]:
from src.pipeline import LesionAnalyzer

analyzer = LesionAnalyzer(strict=False)
print("backbone:", analyzer.model.backbone.describe())
print("trained :", analyzer.is_trained)
if analyzer.model.backbone.is_fallback:
    print("\nNOTE: this is the ImageNet fallback, not PanDerm.")
    print("See README > 'Using PanDerm' to enable it.")

In [ ]:
record = df.iloc[0]
result = analyzer.analyse(load_config().resolve(record["image_path"]),
                          age=record["age"], sex=record["sex"], site=record["site"])

if result.stopped_at_quality_gate:
    print("Stopped at the quality gate:", result.quality.reason)
else:
    print(f"true label : {record['risk_label']}")
    print(f"predicted  : {result.report.risk_level} "
          f"({result.report.confidence:.0%} confident)")
    print(f"recommend  : {result.report.recommendation}")

    fig, axes = plt.subplots(1, 2, figsize=(11, 5))
    axes[0].imshow(result.image)
    axes[0].set_title("photo")
    axes[0].axis("off")
    axes[1].imshow(result.gradcam.overlay)
    axes[1].axis("off")
    ratio = result.gradcam.focus_ratio
    axes[1].set_title(f"Grad-CAM ({ratio:.1f}x more focused than chance)"
                      if ratio else "Grad-CAM")
    plt.tight_layout()

In [ ]:
from IPython.display import Markdown, display

if result.report is not None:
    display(Markdown(result.report.to_markdown()))

## 7. Where to go next

- Download HAM10000 / PAD-UFES-20 and re-run `data/prepare_*.py` for real data.
- `python src/model/train.py`, then `python src/evaluate.py` for real metrics.
- Set `model.use_metadata: false` in the config and retrain, to measure how much
  the age/sex/site metadata actually contributes.
- Enable PanDerm (README > "Using PanDerm") and compare it against the
  fallback backbone.